# Trio-Based Whole Exome Sequencing Analysis
This cleaned notebook contains the final reproducible workflow used for trio WES analysis on the GIAB Ashkenazim Trio.

## 1. Environment Setup

In [2]:
!apt-get update -qq

!apt-get install -y \
bwa \
samtools \
bcftools \
default-jre

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0 libhts3
  libhtscodecs2 libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic cwltool mesa-utils
The following NEW packages will be installed:
  at-spi2-core bcftools bwa default-jre default-jre-headless fonts-dej

In [3]:
!which bwa
!which samtools
!which bcftools

/usr/bin/bwa
/usr/bin/samtools
/usr/bin/bcftools


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Verify Input Files

In [4]:
!ls -lh "/content/drive/MyDrive/Trio_Project" | grep bam

-rw------- 1 root root 1.3G Jun 23 21:18 father.bam
-rw------- 1 root root 2.2M Jun 23 21:19 father.bam.bai
-rw------- 1 root root 1.3G Jun 23 21:24 mother.bam
-rw------- 1 root root 2.1M Jun 23 21:24 mother.bam.bai
-rw------- 1 root root 1.3G Jun 23 21:13 proband.bam
-rw------- 1 root root 2.2M Jun 23 21:14 proband.bam.bai


In [5]:
!ls -lh "/content/drive/MyDrive/Trio_Project" | grep hg38.fa

-rw------- 1 root root 3.1G Jun 22 15:27 hg38.fa
-rw------- 1 root root  21K Jun 23 15:11 hg38.fa.amb
-rw------- 1 root root  22K Jun 23 15:11 hg38.fa.ann
-rw------- 1 root root 3.0G Jun 23 15:11 hg38.fa.bwt
-rw------- 1 root root 766M Jun 23 15:12 hg38.fa.pac
-rw------- 1 root root 1.5G Jun 23 15:12 hg38.fa.sa


## 3. Reference Genome Preparation

In [5]:
!mkdir -p /content/trio


In [7]:
!cp "/content/drive/MyDrive/Trio_Project/hg38.fa" \
/content/trio/

^C


In [8]:
!ls -lh /content/trio/hg38.fa

-rw------- 1 root root 64M Jun 25 18:31 /content/trio/hg38.fa


In [17]:
%%time

!bwa index /content/trio/hg38.fa

[bwa_index] Pack FASTA... 28.31 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=6418572210, availableWord=463634060
[BWTIncConstructFromPacked] 10 iterations done. 99999986 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 199999986 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 299999986 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 399999986 characters processed.
[BWTIncConstructFromPacked] 50 iterations done. 499999986 characters processed.
[BWTIncConstructFromPacked] 60 iterations done. 599999986 characters processed.
[BWTIncConstructFromPacked] 70 iterations done. 699999986 characters processed.
[BWTIncConstructFromPacked] 80 iterations done. 799999986 characters processed.
[BWTIncConstructFromPacked] 90 iterations done. 899999986 characters processed.
[BWTIncConstructFromPacked] 100 iterations done. 999999986 characters processed.
[BWTIncConstructFromPacked] 110 iterations done. 

In [18]:
!ls -lh /content/trio | grep hg38.fa

-rw------- 1 root root 3.1G Jun 23 12:41 hg38.fa
-rw-r--r-- 1 root root  21K Jun 23 14:13 hg38.fa.amb
-rw-r--r-- 1 root root  22K Jun 23 14:13 hg38.fa.ann
-rw-r--r-- 1 root root 3.0G Jun 23 14:12 hg38.fa.bwt
-rw-r--r-- 1 root root 766M Jun 23 14:13 hg38.fa.pac
-rw-r--r-- 1 root root 1.5G Jun 23 15:06 hg38.fa.sa


In [19]:
!cp /content/trio/hg38.fa.* \
"/content/drive/MyDrive/Trio_Project/"

In [8]:
!ls -lh "/content/drive/MyDrive/Trio_Project" | grep hg38.fa

-rw------- 1 root root 3.1G Jun 22 15:27 hg38.fa
-rw------- 1 root root  21K Jun 23 15:11 hg38.fa.amb
-rw------- 1 root root  22K Jun 23 15:11 hg38.fa.ann
-rw------- 1 root root 3.0G Jun 23 15:11 hg38.fa.bwt
-rw------- 1 root root 766M Jun 23 15:12 hg38.fa.pac
-rw------- 1 root root 1.5G Jun 23 15:12 hg38.fa.sa


In [23]:
!samtools faidx /content/drive/MyDrive/Trio_Project/hg38.fa

In [30]:
!/content/gatk-4.6.2.0/gatk CreateSequenceDictionary \
-R /content/drive/MyDrive/Trio_Project/hg38.fa \
-O /content/drive/MyDrive/Trio_Project/hg38.dict

Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar CreateSequenceDictionary -R /content/drive/MyDrive/Trio_Project/hg38.fa -O /content/drive/MyDrive/Trio_Project/hg38.dict
18:46:31.008 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Thu Jun 25 18:46:31 UTC 2026] CreateSequenceDictionary --OUTPUT /content/drive/MyDrive/Trio_Project/hg38.dict --REFERENCE /content/drive/MyDrive/Trio_Project/hg38.fa --TRUNCATE_NAMES_AT_WHITESPACE true --NUM_SEQUENCES 2147483647 --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 2 --MAX_RECORDS_IN_RAM 500000 --CREATE_INDEX false --CREATE_MD5_FILE false --help fals

In [31]:
!ls -lh /content/drive/MyDrive/Trio_Project/hg38.dict

-rw------- 1 root root 56K Jun 25 18:46 /content/drive/MyDrive/Trio_Project/hg38.dict


In [32]:
!samtools faidx /content/drive/MyDrive/Trio_Project/hg38.fa

In [33]:
!ls -lh /content/drive/MyDrive/Trio_Project/hg38.fa.fai

-rw------- 1 root root 19K Jun 25 18:50 /content/drive/MyDrive/Trio_Project/hg38.fa.fai


## 4. Read Alignment (BWA-MEM)

In [21]:
%%time

!bwa mem -t 4 \
-R '@RG\tID:HG002\tSM:HG002\tPL:ILLUMINA' \
/content/trio/hg38.fa \
"/content/drive/MyDrive/Trio_Project/trimmed/proband_R1_paired.fastq.gz" \
"/content/drive/MyDrive/Trio_Project/trimmed/proband_R2_paired.fastq.gz" \
> /content/trio/proband.sam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 177790 sequences (40000427 bp)...
[M::process] read 178746 sequences (40000355 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (4, 74984, 14, 3)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (349, 401, 459)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (129, 679)
[M::mem_pestat] mean and std.dev: (406.64, 81.16)
[M::mem_pestat] low and high boundaries for proper pairs: (19, 789)
[M::mem_pestat] analyzing insert size distribution for orientation RF...
[M::mem_pestat] (25, 50, 75) percentile: (18, 43, 70)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 174)
[M::mem_pestat] mean and std.dev: (36.75, 24.49)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 226)
[M::mem_pestat] skip orientation RR as there ar

In [22]:
%%time

!bwa mem -t 4 \
-R '@RG\tID:HG003\tSM:HG003\tPL:ILLUMINA' \
/content/trio/hg38.fa \
"/content/drive/MyDrive/Trio_Project/trimmed/father_R1_paired.fastq.gz" \
"/content/drive/MyDrive/Trio_Project/trimmed/father_R2_paired.fastq.gz" \
> /content/trio/father.sam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 176278 sequences (40000023 bp)...
[M::process] read 176030 sequences (40000029 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (4, 74383, 10, 4)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (349, 401, 458)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (131, 676)
[M::mem_pestat] mean and std.dev: (406.55, 80.65)
[M::mem_pestat] low and high boundaries for proper pairs: (22, 785)
[M::mem_pestat] analyzing insert size distribution for orientation RF...
[M::mem_pestat] (25, 50, 75) percentile: (2, 13, 64)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 188)
[M::mem_pestat] mean and std.dev: (25.30, 27.77)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 250)
[M::mem_pestat] skip orientation RR as there are

In [23]:
%%time

!bwa mem -t 4 \
-R '@RG\tID:HG004\tSM:HG004\tPL:ILLUMINA' \
/content/trio/hg38.fa \
"/content/drive/MyDrive/Trio_Project/trimmed/mother_R1_paired.fastq.gz" \
"/content/drive/MyDrive/Trio_Project/trimmed/mother_R2_paired.fastq.gz" \
> /content/trio/mother.sam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 174722 sequences (40000097 bp)...
[M::process] read 174714 sequences (40000481 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (12, 74263, 21, 1)
[M::mem_pestat] analyzing insert size distribution for orientation FF...
[M::mem_pestat] (25, 50, 75) percentile: (219, 272, 340)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 582)
[M::mem_pestat] mean and std.dev: (261.64, 55.97)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 703)
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (356, 409, 468)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (132, 692)
[M::mem_pestat] mean and std.dev: (414.64, 82.82)
[M::mem_pestat] low and high boundaries for proper pairs: (20, 804)
[M::mem_pestat] analyzing insert size distribution for orientation RF...
[M::mem_pestat] (25, 50, 75) percen

## 5. BAM Processing & Verification

In [16]:
!find /content/drive/MyDrive/Trio_Project -name "*.bam"

/content/drive/MyDrive/Trio_Project/proband.bam
/content/drive/MyDrive/Trio_Project/father.bam
/content/drive/MyDrive/Trio_Project/mother.bam


In [17]:
!find /content/drive/MyDrive/Trio_Project -name "*.bai"

/content/drive/MyDrive/Trio_Project/proband.bam.bai
/content/drive/MyDrive/Trio_Project/father.bam.bai
/content/drive/MyDrive/Trio_Project/mother.bam.bai


In [18]:
!find /content/drive/MyDrive/Trio_Project -name "hg38.fa"

/content/drive/MyDrive/Trio_Project/hg38.fa


In [19]:
!find /content/drive/MyDrive/Trio_Project -name "*.dict"

In [20]:
!find /content/drive/MyDrive/Trio_Project -name "*.fai"

In [22]:
!samtools --version | head -1

samtools 1.13


## 6. Variant Calling (GATK HaplotypeCaller)

In [25]:
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

In [26]:
!unzip -q gatk-4.6.2.0.zip

In [29]:
!/content/gatk-4.6.2.0/gatk --version

Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar --version
The Genome Analysis Toolkit (GATK) v4.6.2.0
HTSJDK Version: 4.2.0
Picard Version: 3.4.0


In [34]:
!mkdir -p /content/drive/MyDrive/Trio_Project/gvcf

In [35]:
!/content/gatk-4.6.2.0/gatk HaplotypeCaller \
-R /content/drive/MyDrive/Trio_Project/hg38.fa \
-I /content/drive/MyDrive/Trio_Project/proband.bam \
-O /content/drive/MyDrive/Trio_Project/gvcf/proband.g.vcf.gz \
-ERC GVCF

Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R /content/drive/MyDrive/Trio_Project/hg38.fa -I /content/drive/MyDrive/Trio_Project/proband.bam -O /content/drive/MyDrive/Trio_Project/gvcf/proband.g.vcf.gz -ERC GVCF
18:53:46.387 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
18:53:46.783 INFO  HaplotypeCaller - ------------------------------------------------------------
18:53:46.790 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.6.2.0
18:53:46.790 INFO  HaplotypeCaller - For support and documentation go to https://software.broadinstitute.org/gatk/
18:53:46.791 INFO  HaplotypeCaller - Exe

In [36]:
!/content/gatk-4.6.2.0/gatk HaplotypeCaller \
-R /content/drive/MyDrive/Trio_Project/hg38.fa \
-I /content/drive/MyDrive/Trio_Project/father.bam \
-O /content/drive/MyDrive/Trio_Project/gvcf/father.g.vcf.gz \
-ERC GVCF \
2>&1 | tee /content/drive/MyDrive/Trio_Project/logs/father_gvcf.log

tee: /content/drive/MyDrive/Trio_Project/logs/father_gvcf.log: No such file or directory
Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R /content/drive/MyDrive/Trio_Project/hg38.fa -I /content/drive/MyDrive/Trio_Project/father.bam -O /content/drive/MyDrive/Trio_Project/gvcf/father.g.vcf.gz -ERC GVCF
21:57:58.795 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
21:57:59.154 INFO  HaplotypeCaller - ------------------------------------------------------------
21:57:59.159 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.6.2.0
21:57:59.159 INFO  HaplotypeCaller - For support and documentation g

In [37]:
!/content/gatk-4.6.2.0/gatk HaplotypeCaller \
-R /content/drive/MyDrive/Trio_Project/hg38.fa \
-I /content/drive/MyDrive/Trio_Project/mother.bam \
-O /content/drive/MyDrive/Trio_Project/gvcf/mother.g.vcf.gz \
-ERC GVCF \
2>&1 | tee /content/drive/MyDrive/Trio_Project/logs/mother_gvcf.log

tee: /content/drive/MyDrive/Trio_Project/logs/mother_gvcf.log: No such file or directory
Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R /content/drive/MyDrive/Trio_Project/hg38.fa -I /content/drive/MyDrive/Trio_Project/mother.bam -O /content/drive/MyDrive/Trio_Project/gvcf/mother.g.vcf.gz -ERC GVCF
00:59:49.699 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
00:59:50.014 INFO  HaplotypeCaller - ------------------------------------------------------------
00:59:50.020 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.6.2.0
00:59:50.021 INFO  HaplotypeCaller - For support and documentation g

## 7. Joint Genotyping

In [41]:
!/content/gatk-4.6.2.0/gatk CombineGVCFs \
-R /content/drive/MyDrive/Trio_Project/hg38.fa \
-V /content/drive/MyDrive/Trio_Project/gvcf/proband.g.vcf.gz \
-V /content/drive/MyDrive/Trio_Project/gvcf/father.g.vcf.gz \
-V /content/drive/MyDrive/Trio_Project/gvcf/mother.g.vcf.gz \
-O /content/drive/MyDrive/Trio_Project/gvcf/trio_combined.g.vcf.gz \
2>&1 | tee /content/drive/MyDrive/Trio_Project/logs/combine_gvcfs.log

tee: /content/drive/MyDrive/Trio_Project/logs/combine_gvcfs.log: No such file or directory
Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar CombineGVCFs -R /content/drive/MyDrive/Trio_Project/hg38.fa -V /content/drive/MyDrive/Trio_Project/gvcf/proband.g.vcf.gz -V /content/drive/MyDrive/Trio_Project/gvcf/father.g.vcf.gz -V /content/drive/MyDrive/Trio_Project/gvcf/mother.g.vcf.gz -O /content/drive/MyDrive/Trio_Project/gvcf/trio_combined.g.vcf.gz
04:00:12.617 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:00:12.864 INFO  CombineGVCFs - ------------------------------------------------------------
04:00:12.870 INFO  Combin

In [42]:
!/content/gatk-4.6.2.0/gatk GenotypeGVCFs \
-R /content/drive/MyDrive/Trio_Project/hg38.fa \
-V /content/drive/MyDrive/Trio_Project/gvcf/trio_combined.g.vcf.gz \
-O /content/drive/MyDrive/Trio_Project/joint_trio.vcf.gz \
2>&1 | tee /content/drive/MyDrive/Trio_Project/logs/genotype_gvcfs.log

tee: /content/drive/MyDrive/Trio_Project/logs/genotype_gvcfs.log: No such file or directory
Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R /content/drive/MyDrive/Trio_Project/hg38.fa -V /content/drive/MyDrive/Trio_Project/gvcf/trio_combined.g.vcf.gz -O /content/drive/MyDrive/Trio_Project/joint_trio.vcf.gz
05:06:46.137 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:06:46.390 INFO  GenotypeGVCFs - ------------------------------------------------------------
05:06:46.393 INFO  GenotypeGVCFs - The Genome Analysis Toolkit (GATK) v4.6.2.0
05:06:46.393 INFO  GenotypeGVCFs - For support and documentation go

## 8. De Novo Variant Filtering

In [15]:
!bcftools view -h /content/drive/MyDrive/Trio_Project/trio.vcf | tail -5

##bcftools_mergeVersion=1.13+htslib-1.13+ds
##bcftools_mergeCommand=merge -O v -o /content/drive/MyDrive/Trio_Project/trio.vcf /content/drive/MyDrive/Trio_Project/proband.vcf.gz /content/drive/MyDrive/Trio_Project/father.vcf.gz /content/drive/MyDrive/Trio_Project/mother.vcf.gz; Date=Wed Jun 24 17:30:18 2026
##bcftools_viewVersion=1.13+htslib-1.13+ds
##bcftools_viewCommand=view -h /content/drive/MyDrive/Trio_Project/trio.vcf; Date=Thu Jun 25 18:36:21 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	HG002	HG003	HG004


In [46]:
!bcftools view -H /content/drive/MyDrive/Trio_Project/joint_trio.vcf.gz | wc -l

1104551


In [47]:
!bcftools view -H /content/drive/MyDrive/Trio_Project/joint_trio.vcf.gz | head

chr1	39261	.	T	C	39.24	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=25;QD=19.62;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0	./.:.:.:.:.	./.:.:.:.:.
chr1	63735	.	CCTA	C	75.24	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-0.431;DP=3;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=25.68;MQRankSum=0.431;QD=25.08;ReadPosRankSum=-0.967;SOR=0.223	GT:AD:DP:GQ:PL	./.:.:.:.:.	0/1:1,2:3:36:81,0,36	./.:.:.:.:.
chr1	92858	.	G	T	39.24	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=57.5;QD=19.62;SOR=0.693	GT:AD:DP:GQ:PL	./.:.:.:.:.	1/1:0,2:2:6:49,6,0	./.:.:.:.:.
chr1	100876	.	T	C	80.24	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=23;QD=25.36;SOR=0.693	GT:AD:DP:GQ:PGT:PID:PL:PS	1|1:0,2:2:6:1|1:100876_T_C:90,6,0:100876	./.:.:.:.:.:.:.:.	./.:.:.:.:.:.:.:.
chr1	100878	.	A	T	80.24	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=23;QD=28.73;SOR=0.693	GT:AD:DP:GQ:PGT:PID:PL:PS	1|1:0,2:2:6:1|1:100876_T_C:90,6,0:100876	./.:.:.:.:.:.:.:.	./.:.:.:.:.:.:.:.
chr1	101895	.	G	A	39.24	.	AC=2;

In [49]:
!bcftools view \
-s HG002 \
-i 'GT="het" || GT="hom"' \
/content/drive/MyDrive/Trio_Project/joint_trio.vcf.gz \
-Oz \
-o /content/drive/MyDrive/Trio_Project/final_analysis/HG002_called.vcf.gz

In [51]:
!bcftools view \
-i 'GT[0]!="RR" && GT[1]="RR" && GT[2]="RR"' \
/content/drive/MyDrive/Trio_Project/joint_trio.vcf.gz \
-Oz \
-o /content/drive/MyDrive/Trio_Project/final_analysis/denovo_candidates.vcf.gz

In [52]:
!bcftools index \
/content/drive/MyDrive/Trio_Project/final_analysis/denovo_candidates.vcf.gz

In [53]:
!bcftools view -H \
/content/drive/MyDrive/Trio_Project/final_analysis/denovo_candidates.vcf.gz | wc -l

12776


In [54]:
!bcftools view -H \
/content/drive/MyDrive/Trio_Project/final_analysis/denovo_candidates.vcf.gz | head

chr1	267434	.	C	A	36.46	.	AC=2;AF=0.333;AN=6;DP=6;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.167;MQ=35;QD=18.23;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0	0/0:3,0:3:9:0,9,70	0/0:1,0:1:3:0,3,20
chr1	267521	.	T	A	36.27	.	AC=2;AF=0.333;AN=6;DP=6;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.167;MQ=35;QD=18.14;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0	0/0:2,0:2:6:0,6,65	0/0:2,0:2:6:0,6,41
chr1	2064772	.	T	G	32.27	.	AC=1;AF=0.167;AN=6;BaseQRankSum=-0.967;DP=7;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.167;MQ=60;MQRankSum=0;QD=10.76;ReadPosRankSum=0.967;SOR=0.223	GT:AD:DP:GQ:PL	0/1:1,2:3:36:40,0,36	0/0:2,0:2:6:0,6,86	0/0:1,0:1:3:0,3,19
chr1	2079413	.	A	G	37.13	.	AC=2;AF=0.333;AN=6;DP=4;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.167;MQ=60;QD=18.56;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0	0/0:1,0:1:3:0,3,40	0/0:1,0:1:3:0,3,40
chr1	2184152	.	C	A	36.69	.	AC=2;AF=0.333;AN=6;DP=5;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.167;MQ=60;QD=18.34;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0	0/0:1,0:1:3:0,3,19	0/0:2,0:2:6:0,6,41
chr1	2994722	.	G	A	32.

In [56]:
!find /content/drive/MyDrive/Trio_Project -name "denovo_candidates*"

/content/drive/MyDrive/Trio_Project/final_analysis/denovo_candidates.vcf.gz
/content/drive/MyDrive/Trio_Project/final_analysis/denovo_candidates.vcf.gz.csi


In [43]:
!bcftools view -H /content/drive/MyDrive/Trio_Project/trio.vcf | head

chr1	39261	.	T	C	37.32	.	ExcessHet=0;FS=0;MQ=25;QD=18.66;SOR=0.693;DP=2;AF=1;MLEAC=1;MLEAF=0.5;AN=2;AC=2	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0	./.:.:.:.:.	./.:.:.:.:.
chr1	63735	.	CCTA	C	73.6	.	BaseQRankSum=-0.431;ExcessHet=0;FS=0;MQ=25.68;MQRankSum=0.431;QD=24.53;ReadPosRankSum=-0.967;SOR=0.223;DP=3;AF=0.5;MLEAC=1;MLEAF=0.5;AN=2;AC=1	GT:AD:DP:GQ:PL	./.:.:.:.:.	0/1:1,2:3:36:81,0,36	./.:.:.:.:.
chr1	92858	.	G	T	37.32	.	ExcessHet=0;FS=0;MQ=57.5;QD=18.66;SOR=0.693;DP=2;AF=1;MLEAC=1;MLEAF=0.5;AN=2;AC=2	GT:AD:DP:GQ:PL	./.:.:.:.:.	1/1:0,2:2:6:49,6,0	./.:.:.:.:.
chr1	100876	.	T	C	78.32	.	ExcessHet=0;FS=0;MQ=23;QD=25.36;SOR=0.693;DP=2;AF=1;MLEAC=1;MLEAF=0.5;AN=2;AC=2	GT:AD:DP:GQ:PL	1/1:0,2:2:6:90,6,0	./.:.:.:.:.	./.:.:.:.:.
chr1	100878	.	A	T	78.32	.	ExcessHet=0;FS=0;MQ=23;QD=28.73;SOR=0.693;DP=2;AF=1;MLEAC=1;MLEAF=0.5;AN=2;AC=2	GT:AD:DP:GQ:PL	1/1:0,2:2:6:90,6,0	./.:.:.:.:.	./.:.:.:.:.
chr1	101895	.	G	A	37.32	.	ExcessHet=0;FS=0;MQ=22;QD=18.66;SOR=0.693;DP=2;AF=1;MLEAC=1;MLEAF=0.5;AN=2;AC=2	GT:AD:DP

In [47]:
!bcftools view \
-s HG002 \
-i 'GT="het" || GT="hom"' \
/content/drive/MyDrive/Trio_Project/trio.vcf \
-o /content/drive/MyDrive/Trio_Project/HG002_variants.vcf

In [50]:
!bcftools view \
-s HG002 \
/content/drive/MyDrive/Trio_Project/trio.vcf \
-O v \
-o /content/drive/MyDrive/Trio_Project/HG002_only.vcf

In [55]:
!bcftools view \
-s HG002 \
-e 'GT="mis"' \
/content/drive/MyDrive/Trio_Project/trio.vcf \
-O v \
-o /content/drive/MyDrive/Trio_Project/HG002_called.vcf

In [67]:
!bcftools view -H \
/content/drive/MyDrive/Trio_Project/isec_output/0000.vcf | head -10

chr1	39261	.	T	C	37.32	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=25;QD=18.66;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0
chr1	100876	.	T	C	78.32	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=23;QD=25.36;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:90,6,0
chr1	100878	.	A	T	78.32	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=23;QD=28.73;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:90,6,0
chr1	128798	.	C	T	37.32	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=20;QD=18.66;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0
chr1	184413	.	C	A	65.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-0.674;DP=5;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=48.28;MQRankSum=0.431;QD=16.41;ReadPosRankSum=0.674;SOR=0.105	GT:AD:DP:GQ:PL	0/1:2,2:4:73:73,0,74
chr1	185303	.	A	T	37.32	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=25;QD=18.66;SOR=0.693	GT:AD:DP:GQ:PL	1/1:0,2:2:6:49,6,0
chr1	186536	.	C	T	63.32	.	AC=2;AF=1;AN=2;DP=2;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=35.78;QD=31.6

In [68]:
!bcftools view \
-i 'QUAL>=30 && FORMAT/DP>=10 && FORMAT/GQ>=20' \
/content/drive/MyDrive/Trio_Project/isec_output/0000.vcf \
-O v \
-o /content/drive/MyDrive/Trio_Project/denovo_highconf.vcf

In [72]:
!bcftools view -H \
/content/drive/MyDrive/Trio_Project/denovo_highconf.vcf | head -20

chr1	121743475	.	C	A	49.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=0.23;DP=11;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=44.29;MQRankSum=-0.253;QD=4.51;ReadPosRankSum=0.447;SOR=0.283	GT:AD:DP:GQ:PL	0/1:9,2:11:57:57,0,372
chr1	121743479	.	T	G	49.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-0.23;DP=11;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=44.29;MQRankSum=-0.253;QD=4.51;ReadPosRankSum=0.23;SOR=0.283	GT:AD:DP:GQ:PL	0/1:9,2:11:57:57,0,372
chr1	121743502	.	A	T	49.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-0.23;DP=11;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=44.29;MQRankSum=-0.253;QD=4.51;ReadPosRankSum=-1.231;SOR=0.283	GT:AD:DP:GQ:PL	0/1:9,2:11:57:57,0,372
chr1	121743544	.	T	A	56.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=0.903;DP=10;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=44.69;MQRankSum=0.275;QD=5.66;ReadPosRankSum=-0.755;SOR=0.446	GT:AD:DP:GQ:PL	0/1:7,3:10:64:64,0,153
chr1	121775029	.	A	C	61.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-2.823;DP=11;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=30.64;MQRankSum=0.21;QD=6.16;ReadPosRankSum=0;SOR=0.693

## 9. Variant Annotation (ANNOVAR)

## 10. OMIM Mapping

## 11. HPO Mapping

## 12. Candidate Variant Prioritization

## 13. Summary
Pipeline complete: FastQC → Trimmomatic → BWA-MEM → SAMtools → GATK HaplotypeCaller → CombineGVCFs → GenotypeGVCFs → De novo filtering → ANNOVAR → OMIM → HPO → Biological interpretation.